hello world for writing fused kernels

In [1]:
import os; os.environ["TRITON_INTERPRET"] = "1"
import torch, triton, triton.language as tl

Python function → Triton IR → Triton GPU IR (MLIR) → LLVM IR → PTX → cubin    (NVIDIA)

In [ ]:
# H100: 132 SMs + 80 GB HBM
# 132 SMs - CUDA cores, tensor cores), registers, and fast shared memory
# and one large memory
# This defines the program to run on an individual SM
@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
	# BLOCK: elements per program, 1024 etc.
	# so add operation occurs 1024 in one go

	# spits out 0...n-1 of current program
	pid = tl.program_id(0)

	# mem offset
	# 2 * 1024 + [0, ..., 1023] 
	# = [2048, ... ]
	offs = pid * BLOCK + tl.arange(0, BLOCK)

	# n % BLOCK != 0
	# [1, 1, ... 0, 0]
	mask = offs < n

	# load them from GPU memory (DRAM) into fast registers.
	# x_ptr is ADDRESS of "x" tensor in continuous memory
	# offs is offset until all of n (num of tensor elements) is used up
	x = tl.load(x_ptr + offs, mask=mask)
	y = tl.load(y_ptr + offs, mask=mask)

	# can only use triton ops
	# x + y adds 1,024 pairs at once
	# adds entire BLOCK at once
	tl.store(out_ptr + offs, x + y, mask=mask)

In [4]:
# Use kernel at least once to cache it.
def add(x, y):
    # out ptr ssetup
    out = torch.empty_like(x)

    # num of elements
    n = x.numel()

    # split into this many programs (ceil div)
    grid = (triton.cdiv(n, 1024),)

    # parallize programs and call
    add_kernel[grid](x, y, out, n, BLOCK=1024)
    return out

In [5]:
!ls -R ~/.triton/cache

ls: cannot access '/root/.triton/cache': No such file or directory


In [8]:
x = torch.randn(10_000); y = torch.randn_like(x)
torch.allclose(add(x, y), x + y)

True